# 模型评估与推理接口

## 学习目标

区分 loss、accuracy、precision、recall 和 F1，编写不依赖训练状态的批量推理函数，并固定预处理、类别映射和输出格式。

## 概念模型

验证集用于选择模型，测试集用于一次性报告最终结果。推理接口必须复用训练时的预处理，并明确类别索引到业务标签的映射。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3)).eval()
inputs = torch.randn(24, 4)
targets = torch.randint(0, 3, (24,))
loader = DataLoader(TensorDataset(inputs, targets), batch_size=6)
class_names = ['negative', 'neutral', 'positive']
assert len(class_names) == 3
print('classes:', dict(enumerate(class_names)))

### 实验 1：收集预测并计算指标

不要把每个 batch 的 accuracy 直接简单平均；先收集预测和标签，再按全量样本计算指标。

In [ ]:
def predict(model, loader, device='cpu'):
    model = model.to(device).eval()
    predictions, labels, probabilities = [], [], []
    with torch.inference_mode():
        for batch_x, batch_y in loader:
            logits = model(batch_x.to(device))
            probabilities.append(logits.softmax(dim=1).cpu())
            predictions.append(logits.argmax(dim=1).cpu())
            labels.append(batch_y.cpu())
    return torch.cat(predictions), torch.cat(labels), torch.cat(probabilities)

predicted, actual, probabilities = predict(model, loader)
accuracy = (predicted == actual).float().mean().item()
print('shapes:', predicted.shape, actual.shape, probabilities.shape, 'accuracy:', round(accuracy, 4))
assert probabilities.shape == (24, 3) and torch.allclose(probabilities.sum(1), torch.ones(24))

In [ ]:
def confusion_matrix(predicted, actual, classes):
    matrix = torch.zeros(classes, classes, dtype=torch.int64)
    for truth, prediction in zip(actual, predicted):
        matrix[truth, prediction] += 1
    return matrix

matrix = confusion_matrix(predicted, actual, len(class_names))
true_positive = matrix.diag().float()
precision = true_positive / matrix.sum(0).clamp_min(1)
recall = true_positive / matrix.sum(1).clamp_min(1)
f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-8)
print('confusion matrix:\n', matrix)
print('macro precision/recall/F1:', precision.mean().item(), recall.mean().item(), f1.mean().item())

### 实验 2：固定推理契约

生产推理至少要返回类别索引、业务类别名、置信度，并明确模型处于 `eval()` 和 `inference_mode()`。

In [ ]:
def classify_one(model, features, names):
    predicted, _, probabilities = predict(model, DataLoader(TensorDataset(features, torch.zeros(len(features), dtype=torch.long))))
    confidence = probabilities.max(dim=1).values
    return [{'class_id': int(i), 'label': names[int(i)], 'confidence': float(score)} for i, score in zip(predicted, confidence)]

result = classify_one(model, inputs[:2], class_names)
print(result)
assert set(result[0]) == {'class_id', 'label', 'confidence'}

## 检查点

说明为什么类别不平衡时 accuracy 可能误导，以及 precision 和 recall 分别适合关注什么错误。列出推理接口必须固定的三项内容。

## 试一试

构造一个几乎全为同一类别的预测，比较 accuracy 与 macro F1；再把 `class_names` 顺序改错，观察业务结果为何会被错误映射。

## 常见错误与调试

推理时忘记 `eval()`、忘记关闭梯度、训练和推理预处理不一致、类别映射与训练标签顺序不一致、把 softmax 概率当成校准后的真实概率。